## Import neccessary Python packages

In [ ]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox
from onsset import *

## Select the csv file with extracted GIS data to be calibrated

In [ ]:
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('OnSSET', 'Open the input file with extracted GIS data')
input_file = filedialog.askopenfilename()
onsseter = SettlementProcessor(input_file)

onsseter.conditioning()

## Enter key demographic and electrification parameters

In [ ]:
provinces = ['GAZA', 'TETE', 'MAPUTO', 'MANICA', 'INHAMBANE', 'SOFALA', 'NIASSA', 'ZAMBEZIA', 'NAMPULA', 'CABO DELGADO']

### Calibrate population and urban/rural status

In [ ]:
start_year = 2024
pop_start_year = 33244414       ### Write the population in the base year (e.g. 2024)
urban_ratio_start_year = 0.3486 ### Write the urban population population ratio in the base year (e.g. 2024)
num_people_per_hh_urban = 4.7     ### Write the number of people per household in urban areas
num_people_per_hh_rural = 4.5   ### Write the number of people per household  in rural areas

In [ ]:
pop_modelled, urban_modelled = onsseter.calibrate_current_pop_and_urban(pop_start_year, urban_ratio_start_year, num_people_per_hh_rural, num_people_per_hh_urban, start_year)

### Define the household size in each settlement based on the province household size

In [ ]:
df = onsseter.df

In [ ]:
# Here define the number of people per household in each province 
hh_size = {
    'CABO DELGADO': 4.5,
    'ZAMBEZIA': 4.6,
    'SOFALA': 4.9,
    'INHAMBANE': 4.1, 
    'TETE': 4.5,
    'NAMPULA': 4.6,
    'NIASSA': 4.9,
    'GAZA': 4.6,
    'MANICA': 4.9,
    'MAPUTO': 4.3,
}

In [ ]:
df['NumPeoplePerHH'] = 4.6  # National average number of people per household in each province

for p in provinces:
    df.loc[df.Admin_1 == p, 'NumPeoplePerHH'] = hh_size[p]  

In [ ]:
df['FinalElecCode{}'.format(start_year)] = 99
df['ElecPop{}'.format(start_year)] = 0
df['ElecStart'] = 0
df['HHs2024'] = df['Pop{}'.format(start_year)] / df['NumPeoplePerHH']

### Identify mini-grid electrified settlements

In [ ]:
# All settlements where there is a mini-grid (MGDist == 0) are considered mini-grid electrified 
df.loc[df['MGDist'] == 0, 'ElecStart'] = 1
df.loc[df['MGDist'] == 0, 'FinalElecCode{}'.format(start_year)] = 8
df.loc[df['MGDist'] == 0, 'ElecPop{}'.format(start_year)] = df['Pop{}'.format(start_year)]

### Identify grid-electrified settlements

#### First, identify settlements which are directly on the MV line, and has a minimum number of population

In [ ]:
min_pop = 200

df.loc[(df['CurrentMVLineDist'] == 0) & (df['Pop{}'.format(start_year)] > min_pop), 'ElecStart'] = 1
df.loc[(df['CurrentMVLineDist'] == 0) & (df['Pop{}'.format(start_year)] > min_pop), 'FinalElecCode{}'.format(start_year)] = 1

#### Next, identify settlements close to the MV lines that display night-time lights that are also likely electrified

In [ ]:
max_mv_line_distance = 3  # Distance  in km from the existing grid network below which we can assume a settlement could be electrified
min_pop = 500      ### Settlement population above which we can assume that it could be electrified

df.loc[(df['CurrentMVLineDist'] < max_mv_line_distance) & (df['Pop{}'.format(start_year)] > min_pop) & (df['NightLights'] > 0), 'ElecStart'] = 1
df.loc[(df['CurrentMVLineDist'] < max_mv_line_distance) & (df['Pop{}'.format(start_year)] > min_pop) & (df['NightLights'] > 0), 'FinalElecCode{}'.format(start_year)] = 1

#### Finally, calibrate against the number of EDM connected households per province

In [ ]:
# The number of households connected to EDM per province in the start year of the analysis (e.g. 2024)
elec_hhs = {
    'CABO DELGADO': 174035,
    'ZAMBEZIA':  324673,
    'SOFALA':  341560,
    'INHAMBANE':  148373, 
    'TETE':  202169,
    'NAMPULA':  611074,
    'NIASSA':  243761,
    'GAZA':  254848,
    'MANICA':  199753,
    'MAPUTO':  852289,
}

In [ ]:
for p in provinces:
    elec_area_pop = df.loc[(df.Admin_1 == p) & (df['FinalElecCode{}'.format(start_year)] == 1), 'Pop{}'.format(start_year)].sum()
    ratio = min(elec_hhs[p] / (elec_area_pop / hh_size[p]), 1)
    df.loc[(df.Admin_1 == p) & (df['FinalElecCode{}'.format(start_year)] == 1), 'ElecPop{}'.format(start_year)] = df['Pop{}'.format(start_year)] * ratio

### Save as csv

In [ ]:
messagebox.showinfo('OnSSET', 'Browse to the directory and name the calibrated file')
output_file = filedialog.asksaveasfilename()

df['ElecPopCalib'] = df['ElecPop{}'.format(start_year)]
df.to_csv(output_file + '.csv', index=False)